# How a Tree Splits

**Notebook 1 of 5.** A decision tree is a flowchart of yes/no questions the data writes for
itself. Six points here, so every question stays visible.

You will choose a split by hand, then watch scikit-learn choose the same one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

## 1. Six points, two classes

Two features, `x1` and `x2`, and a label that is 0 or 1. That is the whole dataset.

In [ ]:
# six points, two features each: x1 across, x2 up
X = np.array([[2, 2], [2, 1], [2, 3], [1, 2], [1, 1], [3, 3]])
y = np.array([0, 1, 1, 1, 0, 1])

colors = ["#252629", "#FF4A11"]

plt.figure(figsize=(6, 5))

# one scatter call per class, so each gets its own color and legend entry
for label in np.unique(y):
    plt.scatter(
        X[y == label, 0], X[y == label, 1],
        c=colors[label], edgecolor="k", s=160, label=f"class {label}",
    )
plt.xlabel("x1"); plt.ylabel("x2")
plt.xlim(0, 4); plt.ylim(0, 4)
plt.legend(); plt.title("Six labeled points")
plt.show()

No straight line separates these classes. A tree does not need one: it draws **horizontal
and vertical cuts**, one at a time, as many as it likes.

## 2. The question a tree asks

Each node holds some training rows and asks one thing: *is this feature below some
threshold?* Yes goes left, no goes right, and each side is asked again. A node that stops
asking is a **leaf**, and the leaf predicts:

- for **classification**, the **majority class** of the rows that landed there;
- for **regression**, their **mean** target value.

So the only decision is **which question to ask**. The tree tries every threshold on every
feature and keeps whichever makes the two sides purest.

## 3. Gini impurity

**Gini impurity** measures how mixed a node's labels are: the chance of labeling a row
wrongly if you guessed at random, in proportion to how common each label is there.

For $K$ classes, writing $p_k$ for the share of the node's rows that belong to class $k$:

$$G = \sum_{k=1}^{K} p_k (1 - p_k) = 1 - \sum_{k=1}^{K} p_k^2$$

- A **pure** node, holding a single class, scores **0** — there is nothing to get wrong.
- Two classes split evenly score **0.5**, the largest value possible, where a guess is no
  better than a coin toss.

In [ ]:
def gini(labels):
    """Return the Gini impurity of an array of class labels.

    Args:
        labels: 1-D array of integer class labels.

    Returns:
        float: 0.0 for a pure node, 0.5 for an even two-class node.
    """
    if len(labels) == 0:
        return 0.0

    # p_k for every class: the share of the node's rows that carry that label
    proportions = np.bincount(labels) / len(labels)
    return 1 - (proportions ** 2).sum()


print(f"all six points:  {gini(y):.4f}")
print(f"one class only:  {gini(np.array([1, 1, 1])):.4f}")
print(f"an even split:   {gini(np.array([0, 0, 1, 1])):.4f}")

Four 1s and two 0s give $1 - (4/6)^2 - (2/6)^2 = 0.4444$. Every split below is trying to
beat it.

## 4. Scoring a split by hand

A split sends $n_{\text{left}}$ of the node's $n$ rows one way and $n_{\text{right}}$ the
other, leaving two impurities instead of one. They combine into a single score by weighting
each side by the share of rows it holds:

$$G_{\text{split}} = \frac{n_{\text{left}}}{n} G_{\text{left}} + \frac{n_{\text{right}}}{n} G_{\text{right}}$$

The **gain** is what the split bought, $G_{\text{parent}} - G_{\text{split}}$, and the
split with the largest gain wins.

> **Exercise:**
>
> Score the split `x2 <= 2.5`: the Gini of each side, the weighted combination, and the gain
> over the parent's 0.4444.

<details><summary>
Click here for a hint...
</summary>

`left = y[X[:, 1] <= 2.5]`, `right = y[X[:, 1] > 2.5]`, and `len(left) / len(y)` is the left
weight.

</details>

In [ ]:
# your code: score the split x2 <= 2.5 and print its gain

<details><summary>
Click here to compare your answer...
</summary>

Left: 4 rows, **0.5**. Right: 2 rows, **0.0** — pure. Weighted **0.3333**, gain **0.1111**.

The split did not separate the classes. It made one side pure and left the mess on the
other, and that was enough.

</details>

## 5. The greedy search

The tree tries **every threshold on every feature**, scores each as you just did, and takes
the best. Thresholds sit midway between neighboring values, since any cut between the same
two points gives the same two sides.

In [ ]:
def all_splits(X, y):
    """Score every candidate split of a dataset.

    Args:
        X: 2-D array of features.
        y: 1-D array of class labels.

    Returns:
        list of (gain, feature_index, threshold) tuples, best gain first.
    """
    scored = []
    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        # midpoints between neighboring values are the only thresholds worth trying
        for threshold in (values[:-1] + values[1:]) / 2:
            left = y[X[:, feature] <= threshold]
            right = y[X[:, feature] > threshold]

            # each side counts in proportion to how many rows it holds
            weighted = len(left) / len(y) * gini(left) + len(right) / len(y) * gini(right)
            scored.append((gini(y) - weighted, feature, threshold))
    return sorted(scored, reverse=True)


for gain, feature, threshold in all_splits(X, y):
    print(f"x{feature + 1} <= {threshold:.1f}   gain {gain:.4f}")

`x2 <= 2.5` wins at 0.1111, ahead of `x1 <= 2.5` at 0.0444. The tree takes it and repeats
the search **inside each side**.

That is what *greedy* means: the best split available **right now**, with no check on
whether a worse one here would allow a better one below. Fast, readable, and not guaranteed
to find the best tree.

## 6. One node, on its own

`max_depth=1` gives a tree of a single node: one question, two leaves. It is called a
**decision stump**, and it is the smallest a tree can be.

In [ ]:
# a tree allowed exactly one question: the decision stump
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(X, y)

fig, ax = plt.subplots(figsize=(8, 4))
plot_tree(stump, feature_names=["x1", "x2"], class_names=["0", "1"], filled=True, ax=ax)
plt.show()

print(f"training accuracy: {stump.score(X, y):.4f}")

It asks `x2 <= 2.5`, the split the search just chose, and gets **4 of 6** right. The two
leaves are worth comparing:

- the right one holds 2 rows, both class 1 — `gini` **0**, and the prediction is certain;
- the left one holds 2 of each class — `gini` **0.5**, and scikit-learn breaks the tie by
  predicting class 0, so half of those rows are wrong.

That left leaf is why a tree does not stop at one node.

## 7. Letting it grow

The stump stopped because `max_depth=1` told it to. Take the limit off and the same search
repeats inside each side, as deep as it can go.

In [ ]:
# no limits, so the tree keeps splitting until every leaf is pure
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X, y)

print(f"nodes: {tree.tree_.node_count}, depth: {tree.tree_.max_depth}")
print(f"training accuracy: {tree.score(X, y):.4f}")

Nine nodes, depth 3, every training point correct. Nothing stopped it, so it split until
every leaf was pure. **Notebook 02 is about why that is a problem.**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(tree, feature_names=["x1", "x2"], class_names=["0", "1"], filled=True, ax=ax)
plt.show()

Each box holds, top to bottom:

- **the question**, absent in a leaf, which has stopped asking;
- **`gini`**, the impurity of the node — the root shows the 0.4444 you computed;
- **`samples`**, how many training rows are there;
- **`value`**, how they split across the classes, `[class 0, class 1]`;
- **`class`**, what the node would predict.

The root asks `x2 <= 2.5` — the split that won the search.

In [ ]:
print(export_text(tree, feature_names=["x1", "x2"]))

## 8. The boundary the tree drew

One feature at a time means every cut is horizontal or vertical, so the regions are always
rectangles.

In [ ]:
xx, yy = np.meshgrid(np.arange(0, 4, 0.01), np.arange(0, 4, 0.01))

# a prediction for every point of the grid, reshaped back into the grid's shape
grid_predictions = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, grid_predictions, alpha=0.25, colors=colors, levels=[-0.5, 0.5, 1.5])

# one scatter call per class, so each gets its own color and legend entry
for label in np.unique(y):
    plt.scatter(
        X[y == label, 0], X[y == label, 1],
        c=colors[label], edgecolor="k", s=160, label=f"class {label}",
    )
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend(); plt.title("Decision boundary")
plt.show()

Two things to notice.

**Cuts sit halfway between points.** Move one point and the boundary moves with it. Trees
are sensitive to small changes in the data — the weakness bagging exists to fix.

**The model answers everywhere.** The data covers 1 to 3; the model labels the whole square,
confidently. A tree never says *I have not seen anything like this*.

## 9. Entropy, the other criterion

Gini is the default, not the only choice. **Entropy** measures the same mixedness in bits,
with $K$ and $p_k$ meaning exactly what they did above:

$$H = -\sum_{k=1}^{K} p_k \log_2 p_k$$

It is **0** for a pure node and **1** for an even two-class node, and it penalizes mixed
nodes slightly harder than Gini does.

> **Exercise:**
>
> Fit a tree with `criterion="entropy"` and compare its node count, depth and training
> accuracy with the Gini tree.

<details><summary>
Click here for a hint...
</summary>

`DecisionTreeClassifier(criterion="entropy", random_state=42)`, then `tree_.node_count`,
`tree_.max_depth` and `.score(X, y)`.

</details>

In [ ]:
# your code: fit a tree with criterion="entropy" and compare it with the Gini tree

<details><summary>
Click here to compare your answer...
</summary>

**The same tree**: 9 nodes, depth 3, the same questions in the same order.

That is the usual outcome — the two measures rarely disagree, and the effect on accuracy is
tiny when they do. Gini is the default because it is cheaper: no logarithm.

</details>

## Summary

- One **threshold question about one feature** per node; leaves predict the majority class,
  or the mean for regression.
- **Gini** scores how mixed a node is; a split is scored by the **weighted** impurity of its
  sides, and the largest **gain** wins.
- The search is **greedy**: best split available now, no look-ahead.
- Cuts are **axis-aligned**, sit **midway between values**, and the model predicts even where
  it has no data.

Left alone, the tree split until every leaf was pure. Next, in `02_classification_tree.ipynb`:
what that costs on real data.